# Data Cleaning & Processing Module

Reads raw CSVs from `data/raw/` and produces cleaned, feature-engineered files in `data/cleaned/`.

Pipeline steps:
1. **Load** raw price, financial statement, and macro data
2. **Deduplicate** — detect and remove duplicate rows with logging
3. **Missing values** — forward-fill trading gaps; drop only unfillable head rows
4. **Data-type normalisation** — dates parsed, numerics coerced
5. **Outlier detection** — flag single-day moves > ±50% (possible splits / data errors)
6. **Feature engineering** — daily returns, 7-day & 30-day MAs, 30-day volatility, Bollinger Bands
7. **Save** cleaned files

In [ ]:
import pandas as pd
import numpy as np
import warnings
import logging
import os

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger()

RAW_DIR     = "data/raw"
CLEANED_DIR = "data/cleaned"
os.makedirs(CLEANED_DIR, exist_ok=True)

print("Data Cleaning & Processing Pipeline")
print("=" * 50)

## 1. Load Raw Price Data

print("Loading raw price data...")
price_df = pd.read_csv(f"{RAW_DIR}/sp500_prices.csv")

# Guard: 'Ticker' missing means the CSV was saved with the wrong stack level (yfinance 1.x bug)
if "Ticker" not in price_df.columns:
    raise ValueError(
        f"'Ticker' column not found in sp500_prices.csv.\n"
        f"Actual columns: {price_df.columns.tolist()}\n"
        "The CSV was generated by run_collect.py with the wrong yfinance 1.x stack level.\n"
        "Re-run 'run_collect.py' (now fixed) and retry this notebook."
    )

# Normalise date — strip timezone if present
price_df["Date"] = pd.to_datetime(price_df["Date"], utc=True).dt.tz_localize(None)
price_df = price_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print(f"Loaded : {price_df.shape[0]:,} rows | {price_df['Ticker'].nunique()} tickers")
print(f"Range  : {price_df['Date'].min().date()} → {price_df['Date'].max().date()}")
price_df.head()

## 2. Price Data Cleaning

In [ ]:
import pandas as pd

print("=== Price Data Cleaning ===\n")
initial_rows = len(price_df)

# --- 2a. Duplicate removal ---
# Optimized: Dropping duplicates upfront is fast; no need to count them first unless strictly required.
# If counting is necessary, price_df.duplicated() is fine, but we can combine actions.
before_dedup = len(price_df)
price_df = price_df.drop_duplicates(subset=["Ticker", "Date"])
dupes = before_dedup - len(price_df)

if dupes:
    log.info(f"[DEDUP]   Removed {dupes} duplicate rows")
else:
    log.info("[DEDUP]   No duplicates found")

# --- 2b. Enforce numeric types ---
# Optimized: Downcast to save memory/speed up downstream operations. 
# Also vectorizes the conversion across columns efficiently.
numeric_cols = ["Open", "High", "Low", "Close", "Volume"]
existing_numeric_cols = [col for col in numeric_cols if col in price_df.columns]
if existing_numeric_cols:
    price_df[existing_numeric_cols] = price_df[existing_numeric_cols].apply(
        pd.to_numeric, errors="coerce"
    )

# --- 2c. Forward-fill missing business days per ticker ---
# OPTIMIZATION: Avoid the Python `for` loop completely. 
# Generate a MultiIndex of all (Ticker x Business Days) combinations.
all_bdates = pd.bdate_range(price_df["Date"].min(), price_df["Date"].max())
unique_tickers = price_df["Ticker"].unique()

# Create a complete grid of all dates for all tickers
full_index = pd.MultiIndex.from_product(
    [unique_tickers, all_bdates], names=["Ticker", "Date"]
)

# Reindex and forward-fill on the index levels directly (Highly optimized in Pandas)
price_df = (
    price_df.set_index(["Ticker", "Date"])
    .reindex(full_index)
    .groupby(level="Ticker", sort=False)
    .ffill()
    .reset_index()
)

filled = len(price_df) - (initial_rows - dupes)
log.info(f"[FILL]    Forward-filled {filled:,} missing business-day entries")

# --- 2d. Drop rows where Close is still NaN (start of series) ---
# Left as is, since dropna on a subset is already heavily optimized in C.
null_close = price_df["Close"].isna().sum()
if null_close:
    price_df = price_df.dropna(subset=["Close"])
    log.info(f"[DROP]    Removed {null_close:,} rows with no Close price")

# --- 2e. Outlier flagging: single-day return > ±50% ---
# OPTIMIZATION: price_df is already implicitly sorted by Ticker and Date 
# due to the MultiIndex.from_product structure above. We can skip sorting entirely.
pct_chg = price_df.groupby("Ticker", sort=False)["Close"].pct_change()
price_df["is_outlier"] = pct_chg.abs() > 0.50
n_outliers = price_df["is_outlier"].sum()
log.info(f"[OUTLIER] Flagged {n_outliers} entries with single-day move > ±50%")

if n_outliers:
    display(price_df[price_df["is_outlier"]][["Date", "Ticker", "Close"]].head(10))

print(f"\nPrice data after cleaning: {price_df.shape[0]:,} rows")

## 3. Feature Engineering

In [ ]:
import numpy as np
import pandas as pd

print("=== Feature Engineering ===\n")

# --- 1. Preparation ---
# Ensure sorting is done upfront. This is crucial for correct rolling calculations.
price_df = price_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

# Setting the index to Ticker and Date allows us to use vectorized .rolling() 
# without breaking the dataframe structure or dropping columns.
price_df = price_df.set_index(["Ticker", "Date"])
g = price_df.groupby(level="Ticker", sort=False)

# --- 2. Feature Engineering (Vectorized) ---

# Daily return
price_df["daily_return"] = g["Close"].pct_change().round(6)

# Regroup on 'daily_return' since it was just added to the dataframe
g_ret = price_df.groupby(level="Ticker", sort=False)["daily_return"]
g_close = price_df.groupby(level="Ticker", sort=False)["Close"]

# Rolling moving averages
# Passing window directly to g.rolling() utilizes optimized C-level code
price_df["ma7"] = g_close.rolling(window=7, min_periods=1).mean().values.round(4)
price_df["ma30"] = g_close.rolling(window=30, min_periods=1).mean().values.round(4)

# Annualised 30-day rolling volatility
price_df["volatility_30d"] = (
    (g_ret.rolling(window=30, min_periods=5).std() * np.sqrt(252))
    .values.round(6)
)

# Bollinger Bands — 20-day, ±2σ
bb_mid = g_close.rolling(window=20, min_periods=1).mean().values
bb_std = g_close.rolling(window=20, min_periods=1).std().values

price_df["bb_mid"] = np.round(bb_mid, 4)
price_df["bb_upper"] = np.round(bb_mid + 2 * bb_std, 4)
price_df["bb_lower"] = np.round(bb_mid - 2 * bb_std, 4)

# Cumulative return
# Fast vectorized cumprod; fillna(0) ensures the first row handles correctly
price_df["cum_return"] = (
    (1 + price_df["daily_return"].fillna(0))
    .groupby(level="Ticker", sort=False)
    .cumprod() - 1
).round(6)

# Reset index to restore original structure
price_df = price_df.reset_index()

# --- 3. Output ---
print("Engineered features added:")
for feat, desc in [
    ("daily_return", "% change in Close"),
    ("ma7", "7-day rolling moving average"),
    ("ma30", "30-day rolling moving average"),
    ("volatility_30d", "30-day annualised volatility"),
    ("bb_upper/mid/lower", "Bollinger Bands (20-day, ±2σ)"),
    ("cum_return", "Cumulative return from first date"),
]:
    print(f"  {feat:<20} — {desc}")

print(f"\nFinal price shape: {price_df.shape}")
price_df[
    ["Date", "Ticker", "Close", "daily_return", "ma7", "ma30", "volatility_30d"]
].tail(8)

## 6. Save Cleaned Data

print("=== Saving Cleaned Data ===\n")

def _save(df: pd.DataFrame, filename: str):
    if df.empty:
        return
    path = f"{CLEANED_DIR}/{filename}"
    df.to_csv(path, index=False)
    print(f"  {filename:<30} — {df.shape[0]:,} rows saved")

_save(price_df,    "prices_clean.csv")
_save(income_df,   "income_clean.csv")
_save(balance_df,  "balance_clean.csv")
_save(cashflow_df, "cashflow_clean.csv")
_save(macro_df,    "macro_clean.csv")

print(f"\nAll cleaned files written to {CLEANED_DIR}/")